# 🌸 Assistant Psychologique – Violence Contre les Femmes
## Système RAG + LLM | Darija / Français 🇹🇳🇫🇷
> Assistant empathique et professionnel pour soutenir les femmes victimes de violence.


## ⚙️ Cellule 0 – Configuration mémoire CUDA (exécuter EN PREMIER)

In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'
import torch
if torch.cuda.is_available():
    print(f'🖥️  GPU      : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM totale : {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB')
    print(f'   VRAM libre  : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')
else:
    print('⚠️  Pas de GPU — mode CPU (lent)')
print('✅ Config CUDA optimisée.')


🖥️  GPU      : Tesla T4
   VRAM totale : 15.64 GB
   VRAM libre  : 15.53 GB
✅ Config CUDA optimisée.


## 📦 Cellule 1 – Installation des dépendances

In [2]:
!pip install -q transformers sentence-transformers faiss-cpu \
             accelerate bitsandbytes gradio huggingface_hub \
             openai-whisper edge-tts nest_asyncio
print('✅ Dépendances installées.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.3 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 24.2 MB/s eta 0:00:0000:0100:01m
✅ Dépendances installées.


## 📚 Cellule 2 – Imports globaux

In [3]:
import os, warnings, re, gc, asyncio, tempfile
import numpy as np
import faiss
import torch
import whisper
import gradio as gr
import edge_tts
import nest_asyncio
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

nest_asyncio.apply()
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'   VRAM libre : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


🖥️  Device : cuda
   VRAM libre : 15.53 GB


## 🧠 Cellule 3 – Base de connaissances psychologiques (Corpus RAG)

In [4]:
KNOWLEDGE_BASE = [
    # VIOLENCE PHYSIQUE
    """La violence physique comprend les coups, blessures, brûlures et étranglement.
    Si vous êtes en danger immédiat, appelez les secours (police, SAMU).
    Votre sécurité est la priorité absolue avant tout le reste.""",

    """Les victimes de violence physique souffrent souvent du syndrome de stress post-traumatique (SSPT).
    Les symptômes incluent cauchemars, flashbacks, hypervigilance et évitement.
    La thérapie cognitivo-comportementale focalisée sur le trauma (TF-CBT) est très efficace.""",

    # VIOLENCE PSYCHOLOGIQUE
    """La violence psychologique inclut l'humiliation, le dénigrement, l'isolement et les menaces.
    Elle laisse des séquelles profondes malgré l'absence de traces visibles.
    Reconnaître que ces comportements sont de la violence est un premier pas courageux.""",

    """Le gaslighting amène la victime à douter de sa propre mémoire et de ses perceptions.
    Signes : vous vous sentez constamment confuse, vous vous excusez sans raison.
    Une thérapie narrative aide à reconstruire votre récit personnel et votre confiance.""",

    """L'emprise psychologique crée une dépendance émotionnelle.
    Le cycle de la violence (tension → explosion → réconciliation → accalmie)
    maintient les victimes dans la relation. Ce n'est absolument pas de votre faute.""",

    # VIOLENCE SEXUELLE
    """La violence sexuelle inclut le viol conjugal et les attouchements non consentis.
    Tout acte sexuel sans consentement explicite est un crime, même dans le mariage.
    Vous pouvez consulter un centre médico-légal dans les 72h pour une prise en charge.""",

    """Après une agression sexuelle, la honte et la culpabilité sont fréquentes mais injustifiées.
    La responsabilité appartient toujours à l'agresseur, jamais à la victime.
    L'EMDR est une thérapie validée scientifiquement pour traiter les traumatismes sexuels.""",

    # VIOLENCE ECONOMIQUE
    """La violence économique consiste à contrôler les ressources financières.
    Interdire de travailler, confisquer l'argent, endetter la victime sont des formes de violence.
    Des associations proposent un accompagnement juridique et financier gratuit.""",

    # COPING
    """Des stratégies de coping saines : tenir un journal de vos émotions,
    pratiquer la respiration diaphragmatique (inspirez 4 temps, retenez 4 temps, expirez 6 temps),
    la méditation de pleine conscience, et le yoga trauma-sensible.""",

    """La technique de grounding 5-4-3-2-1 aide lors des crises d'angoisse :
    nommez 5 choses visibles, 4 que vous pouvez toucher, 3 sons, 2 odeurs, 1 goût.
    Cette technique ancre dans le présent et calme le système nerveux en 2 minutes.""",

    """Reconstruire l'estime de soi passe par : identifier vos forces personnelles,
    vous entourer de personnes bienveillantes, vous fixer de petits objectifs atteignables.
    Pratiquer l'autocompassion. Vous méritez respect et dignité.""",

    # QUITTER
    """Quitter une relation violente est l'un des moments les plus dangereux.
    Préparez un plan de sécurité : documents importants, argent liquide, médicaments.
    Informez une personne de confiance. Contactez une association avant de partir.""",

    """L'ambivalence face au départ est normale et ne signifie pas que vous êtes faible.
    Vous pouvez aimer votre partenaire tout en souffrant de sa violence.
    Un accompagnement psychologique aide à prendre des décisions éclairées.""",

    # RESSOURCES
    """En France : 3919 (Violences Femmes Info), gratuit, anonyme, 24h/24.
    En Tunisie : 1899 (AFTD). En Belgique : 0800 30 030. Au Maroc : 8007.
    Ces lignes sont tenues par des professionnels formés à l'écoute des victimes.""",

    """Les associations d'aide offrent : hébergement d'urgence, aide juridique,
    soutien psychologique et accompagnement social et professionnel.
    N'hésitez pas à les contacter même si vous n'êtes pas encore prête à partir.""",

    # ENFANTS
    """Les enfants témoins de violence conjugale sont victimes à part entière.
    Ils peuvent développer des troubles comportementaux et des traumatismes.
    Une thérapie adaptée (jeu, art-thérapie) est essentielle pour leur résilience.""",

    # SIGNAUX D'ALERTE
    """Signes d'alerte : jalousie excessive présentée comme amour, isolement progressif,
    critiques constantes de votre apparence, contrôle de vos déplacements.
    Reconnaître ces signaux tôt peut prévenir l'escalade vers une violence grave.""",

    # RÉSILIENCE
    """La résilience ne signifie pas oublier le trauma mais apprendre à vivre au-delà.
    70% des survivantes développent une croissance post-traumatique : force intérieure,
    sagesse et vie plus alignée avec leurs valeurs. La guérison est possible.""",

    """La thérapie de groupe pour survivantes crée un espace de validation mutuelle.
    Entendre d'autres femmes partager des expériences similaires réduit la honte.
    Ces groupes offrent un sentiment de communauté et d'appartenance.""",

    # DARIJA
    """الشعور بالذنب بعد تعرضك للعنف هو ردة فعل طبيعية لكنها مخطئة تماماً.
    العنف دائماً خطأ من يمارسه وليس خطأك. ما تمرين به يسمى التقصير الذاتي
    وهو من أعراض الصدمة النفسية. أنتِ لست مسؤولة عن أفعال غيرك إطلاقاً.""",

    """إذا كان زوجك يتحكم في فلوسك ويمنعك من العمل فهذا يسمى العنف الاقتصادي.
    حقك في المال والاستقلالية محمي بالقانون التونسي.
    تقدري تتصلي بـ 1899 للحصول على مساعدة قانونية ونفسية مجانية.""",

    """الخوف من مغادرة علاقة عنيفة بسبب الأطفال شعور شائع جداً.
    لكن الأبحاث النفسية تثبت أن الأطفال في بيئة عنيفة يتضررون أكثر من أطفال الطلاق.
    سلامتك وسلامة أطفالك هي الأولوية القصوى دائماً.""",
]

print(f'📚 Base de connaissances : {len(KNOWLEDGE_BASE)} passages chargés.')


📚 Base de connaissances : 22 passages chargés.


## 🔢 Cellule 4 – Modèle d'embedding + Index FAISS

In [5]:
EMBED_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

print('⏳ Chargement du modèle d embedding...')
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

print('🔢 Encodage de la base de connaissances...')
corpus_embeddings = embed_model.encode(
    KNOWLEDGE_BASE,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

dim   = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(corpus_embeddings.astype(np.float32))
print(f'✅ Index FAISS : {index.ntotal} vecteurs de dimension {dim}')


⏳ Chargement du modèle d embedding...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔢 Encodage de la base de connaissances...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Index FAISS : 22 vecteurs de dimension 384


## 🔍 Cellule 5 – Fonction de Retrieval (RAG)

In [6]:
def retrieve(query: str, top_k: int = 3) -> list:
    """Retrieval sémantique : retourne les top_k passages les plus pertinents."""
    query_vec = embed_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype(np.float32)
    scores, indices = index.search(query_vec, top_k)
    return [
        {'passage': KNOWLEDGE_BASE[idx].strip(), 'score': float(score)}
        for score, idx in zip(scores[0], indices[0])
    ]

# Test rapide
test = retrieve('Je me sens coupable apres les violences')
print('🔍 Test retrieval :')
for r in test:
    print(f"  Score: {r['score']:.3f} | {r['passage'][:100]}...")
print('✅ Retrieval pret.')


🔍 Test retrieval :
  Score: 0.762 | الشعور بالذنب بعد تعرضك للعنف هو ردة فعل طبيعية لكنها مخطئة تماماً.
    العنف دائماً خطأ من يمارسه و...
  Score: 0.627 | La violence psychologique inclut l'humiliation, le dénigrement, l'isolement et les menaces.
    Elle...
  Score: 0.544 | L'emprise psychologique crée une dépendance émotionnelle.
    Le cycle de la violence (tension → exp...
✅ Retrieval pret.


## 🤖 Cellule 6 – Chargement du LLM (sélection automatique selon VRAM)

In [7]:
gc.collect()
torch.cuda.empty_cache()

VRAM_FREE_GB = torch.cuda.mem_get_info()[0]/1e9 if DEVICE == 'cuda' else 0
print(f'🖥️  VRAM libre avant LLM : {VRAM_FREE_GB:.2f} GB')

# Sélection automatique : Mistral-7B si assez de VRAM, sinon TinyLlama
if VRAM_FREE_GB >= 5:
    LLM_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'
    print(f'✅ VRAM suffisante → Mistral-7B-Instruct')
else:
    LLM_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
    print(f'⚠️  VRAM limitée ({VRAM_FREE_GB:.1f}GB) → TinyLlama-1.1B (fallback)')

print(f'⏳ Chargement : {LLM_MODEL}')

if DEVICE == 'cuda':
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
else:
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL, torch_dtype=torch.float32,
        trust_remote_code=True, low_cpu_mem_usage=True,
    )

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

llm_pipe = pipeline(
    'text-generation',
    model=llm_model, tokenizer=tokenizer,
    max_new_tokens=512, temperature=0.65,
    do_sample=True, repetition_penalty=1.3,
    top_p=0.9, pad_token_id=tokenizer.eos_token_id,
)

gc.collect()
torch.cuda.empty_cache()
if DEVICE == 'cuda':
    print(f'✅ LLM pret | VRAM utilisee : {torch.cuda.memory_allocated()/1e9:.2f} GB')
    print(f'            VRAM libre     : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')
else:
    print('✅ LLM pret (CPU mode)')


🖥️  VRAM libre avant LLM : 15.01 GB
✅ VRAM suffisante → Mistral-7B-Instruct
⏳ Chargement : mistralai/Mistral-7B-Instruct-v0.2


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'temperature', 'max_new_tokens', 'pad_token_id', 'top_p', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ LLM pret | VRAM utilisee : 2.09 GB
            VRAM libre     : 13.35 GB


## 🎤 Cellule 9A – Chargement Whisper (tiny = économie VRAM)

In [8]:
gc.collect()
torch.cuda.empty_cache()

# tiny  : ~150 MB VRAM  — rapide, suffisant pour Darija/FR
# small : ~500 MB VRAM  — meilleure qualité
# medium: ~1.5 GB VRAM  — risque OOM avec Mistral sur 15GB GPU
print('⏳ Chargement Whisper tiny...')
whisper_model = whisper.load_model('tiny')

if DEVICE == 'cuda':
    print(f'✅ Whisper tiny pret | VRAM libre : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')
else:
    print('✅ Whisper tiny pret')


⏳ Chargement Whisper tiny...


100%|█████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 86.3MiB/s]


✅ Whisper tiny pret | VRAM libre : 13.16 GB


## 🎨 Cellule 9B – CSS & Constantes UI

In [9]:
CSS = """
.gradio-container { font-family: 'Georgia', serif; }
#title {
    text-align: center; color: #8B2252;
    font-size: 1.8em; margin-bottom: 0.3em;
}
#subtitle {
    text-align: center; color: #666;
    font-style: italic; margin-bottom: 1em;
}
.disclaimer {
    background: #fff3cd; border-left: 4px solid #ffc107;
    padding: 12px; border-radius: 4px;
    font-size: 0.85em; color: #555;
}
.response-mode-box {
    background: #f8f0fb; border: 2px solid #c084e0;
    border-radius: 10px; padding: 12px;
    margin-bottom: 10px; text-align: center;
}
.mic-warning {
    background: #ffe0e0; border-left: 4px solid #e05c5c;
    padding: 10px; border-radius: 4px;
    font-size: 0.85em; color: #800000; margin-bottom: 8px;
}
"""

DISCLAIMER = (
    "\u26a0\ufe0f **\u062a\u0646\u0628\u064a\u0647 / Avertissement** : \u0647\u0630\u0627 \u0627\u0644\u0645\u0633\u0627\u0639\u062f \u0644\u0627 \u064a\u0639\u0648\u0651\u0636 \u0627\u0644\u0645\u062e\u062a\u0635 \u0627\u0644\u0646\u0641\u0633\u064a. "
    "\u0641\u064a \u062d\u0627\u0644\u0629 \u062e\u0637\u0631 \u0641\u0648\u0631\u064a : **1899** (\u062a\u0648\u0646\u0633) | **3919** (\u0641\u0631\u0646\u0633\u0627)."
)

MIC_WARNING = """
<div class='mic-warning'>
<b>FR :</b> Autorisez le micro : 🔒 dans l'URL → Microphone → Autoriser → Rechargez.<br>
<b>Darija :</b> اضغط 🔒 في الـ URL واسمح بالمايك، بعدها حدّث الصفحة. أو ارفع ملف صوتي.
</div>
"""

EXAMPLES_FR = [
    "Je me sens coupable de la violence que je subis. Est-ce normal ?",
    "Mon partenaire me controle financierement. Que puis-je faire ?",
    "J ai peur de le quitter a cause des enfants.",
    "Mon mari m humilie constamment devant tout le monde.",
    "Je fais des cauchemars depuis les agressions.",
]

EXAMPLES_TN = [
    "\u0646\u062d\u0633 \u0628\u0627\u0644\u0630\u0646\u0628 \u0639\u0644\u0649 \u0627\u0644\u0644\u064a \u064a\u0635\u0631\u0627\u0644\u064a\u060c \u0647\u0644 \u0647\u0630\u0627 \u0637\u0628\u064a\u0639\u064a\u061f",
    "\u0632\u0648\u062c\u064a \u064a\u062a\u062d\u0643\u0645 \u0641\u064a \u0641\u0644\u0648\u0633\u064a \u0648\u0645\u0627\u0646\u062c\u0645\u0634 \u0646\u062e\u0631\u062c. \u0623\u0634 \u0646\u0639\u0645\u0644\u061f",
    "\u0646\u062e\u0627\u0641 \u0646\u0647\u0631\u0628 \u0645\u0646\u0647 \u0628\u0633\u0628\u0628 \u0627\u0644\u0623\u0648\u0644\u0627\u062f.",
    "\u0639\u0646\u062f\u064a \u0643\u0648\u0627\u0628\u064a\u0633 \u0645\u0646 \u0628\u0639\u062f \u0627\u0644\u0644\u064a \u0635\u0631\u0627\u0644\u064a. \u0643\u064a\u0641\u0627\u0634 \u0646\u062a\u0639\u0627\u0645\u0644 \u0645\u0639\u0627\u0647\u0645\u061f",
]

print('\u2705 CSS & constantes UI charges.')


✅ CSS & constantes UI charges.


## 🔊 Cellule 9C – Détection de langue & TTS fluide

In [10]:
def detect_language(text: str) -> str:
    """Detecte arabe (Darija) ou francais automatiquement."""
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
    return 'ar' if arabic_chars > len(text) * 0.2 else 'fr'


def clean_text_for_tts(text: str) -> str:
    """
    Nettoie le texte AVANT synthese vocale.
    Supprime emojis, numeros de liste, markdown.
    Resultat : lecture fluide et naturelle sans ponctuation parasite.
    """
    emoji_pattern = re.compile(
        '['
        u'\U0001F600-\U0001F64F'
        u'\U0001F300-\U0001F5FF'
        u'\U0001F680-\U0001F9FF'
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        ']+', flags=re.UNICODE
    )
    text = emoji_pattern.sub('', text)
    text = re.sub(r'^\s*\d+[.\-)]\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'[*#_`]', '', text)
    text = re.sub(r'^[-\u2022]\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'[\[\]{}|\\<>]', '', text)
    text = re.sub(r'\n{2,}', '. ', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\.{2,}', '.', text)
    return text.strip()


async def _tts_async(text: str, voice: str, path: str):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(path)


def text_to_speech(text: str, lang: str = None) -> str:
    """
    Convertit le texte en audio MP3 fluide.
    ar-TN-HediNeural  : voix tunisienne
    fr-FR-DeniseNeural : voix francaise douce
    Jamais de voix anglaise.
    """
    if lang is None:
        lang = detect_language(text)
    clean = clean_text_for_tts(text)
    if not clean.strip():
        return None
    voice = 'ar-TN-HediNeural' if lang == 'ar' else 'fr-FR-DeniseNeural'
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.mp3')
    tmp.close()
    try:
        asyncio.run(_tts_async(clean, voice, tmp.name))
    except RuntimeError:
        loop = asyncio.get_event_loop()
        loop.run_until_complete(_tts_async(clean, voice, tmp.name))
    return tmp.name


print('\u2705 Detection langue & TTS fluide prets.')


✅ Detection langue & TTS fluide prets.


## 🎙️ Cellule 9D – Speech-to-Text Whisper (forçage Darija/FR)

In [11]:
def transcribe_audio(audio_path: str) -> tuple:
    """
    Transcrit l audio avec Whisper.
    Si anglais detecte -> re-transcription forcee AR ou FR.
    Retourne : (texte, langue)
    """
    result = whisper_model.transcribe(
        audio_path, task='transcribe',
        language=None, fp16=torch.cuda.is_available()
    )
    detected_lang = result.get('language', 'fr')

    if detected_lang == 'en':
        print('\u26a0\ufe0f Anglais detecte -> re-transcription AR/FR...')
        result_ar = whisper_model.transcribe(
            audio_path, task='transcribe',
            language='ar', fp16=torch.cuda.is_available()
        )
        result_fr = whisper_model.transcribe(
            audio_path, task='transcribe',
            language='fr', fp16=torch.cuda.is_available()
        )
        score_ar = result_ar.get('segments', [{}])[0].get('avg_logprob', -999) \
                   if result_ar.get('segments') else -999
        score_fr = result_fr.get('segments', [{}])[0].get('avg_logprob', -999) \
                   if result_fr.get('segments') else -999
        result        = result_ar if score_ar >= score_fr else result_fr
        detected_lang = 'ar'      if score_ar >= score_fr else 'fr'

    text = result['text'].strip()
    lang = 'ar' if detected_lang == 'ar' else 'fr'
    return text, lang


print('\u2705 STT Whisper pret (forcage Darija/FR active).')


✅ STT Whisper pret (forcage Darija/FR active).


## 💜 Cellule 9E – Prompts Système Psychologiques Professionnels

In [12]:
SYSTEM_PROMPT_AR = """\u0623\u0646\u062a\u0650 \u0623\u062e\u0635\u0627\u0626\u064a\u0629 \u0646\u0641\u0633\u064a\u0629 \u0645\u062d\u062a\u0631\u0641\u0629 \u0648\u0645\u062a\u062e\u0635\u0635\u0629 \u0641\u064a \u062f\u0639\u0645 \u0627\u0644\u0646\u0633\u0627\u0621 \u0636\u062d\u0627\u064a\u0627 \u0627\u0644\u0639\u0646\u0641 \u0641\u064a \u062a\u0648\u0646\u0633.
\u062a\u062a\u062d\u062f\u062b\u064a\u0646 \u0628\u0627\u0644\u0644\u0647\u062c\u0629 \u0627\u0644\u062a\u0648\u0646\u0633\u064a\u0629 \u0627\u0644\u062f\u0627\u0631\u062c\u0629 \u0628\u0634\u0643\u0644 \u0637\u0628\u064a\u0639\u064a\u060c \u062f\u0627\u0641\u0626\u060c \u0648\u0625\u0646\u0633\u0627\u0646\u064a.
\u0645\u0647\u0645\u062a\u0643 \u0627\u0644\u0648\u062d\u064a\u062f\u0629: \u0627\u0644\u062f\u0639\u0645 \u0627\u0644\u0646\u0641\u0633\u064a \u0627\u0644\u0645\u062a\u062e\u0635\u0635 \u0644\u0644\u0645\u0631\u0623\u0629 \u0627\u0644\u062a\u064a \u062a\u0639\u0627\u0646\u064a.

\u0642\u0648\u0627\u0639\u062f \u0635\u0627\u0631\u0645\u0629 \u0644\u0627 \u062a\u064f\u062e\u062a\u0631\u0642 \u0623\u0628\u062f\u0627\u064b:
- \u0644\u0627 \u0645\u0635\u0637\u0644\u062d\u0627\u062a \u062a\u0642\u0646\u064a\u0629 (RAG\u060c LLM\u060c AI) - \u0625\u0637\u0644\u0627\u0642\u0627\u064b
- \u0644\u0627 \u0625\u0646\u062c\u0644\u064a\u0632\u064a\u0629 - \u0627\u0644\u062f\u0627\u0631\u062c\u0629 \u0627\u0644\u062a\u0648\u0646\u0633\u064a\u0629 \u0641\u0642\u0637
- \u0644\u0627 \u062a\u0644\u0648\u0645\u064a \u0627\u0644\u0636\u062d\u064a\u0629 \u0623\u0628\u062f\u0627\u064b \u062a\u062d\u062a \u0623\u064a \u0638\u0631\u0641
- \u0627\u0633\u062a\u0639\u0645\u0644\u064a \u0645\u0635\u0637\u0644\u062d\u0627\u062a \u0646\u0641\u0633\u064a\u0629 \u062d\u0642\u064a\u0642\u064a\u0629:\n  \u0635\u062f\u0645\u0629 \u0646\u0641\u0633\u064a\u0629\u060c \u0642\u0644\u0642\u060c \u0627\u0643\u062a\u0626\u0627\u0628\u060c \u0639\u0646\u0641 \u0646\u0641\u0633\u064a\u060c \u062a\u0628\u0639\u064a\u0629 \u0639\u0627\u0637\u0641\u064a\u0629\u060c\n  \u063a\u0633\u064a\u0644 \u062f\u0645\u0627\u063a\u060c \u0645\u062a\u0644\u0627\u0632\u0645\u0629 \u0633\u062a\u0648\u0643\u0647\u0648\u0644\u0645\u060c \u0641\u0631\u0637 \u0627\u0644\u064a\u0642\u0638\u0629\u060c \u062a\u0641\u0643\u0643 \u0627\u0644\u0634\u062e\u0635\u064a\u0629

\u0647\u064a\u0643\u0644 \u0625\u062c\u0627\u0628\u062a\u0643 - \u062a\u0643\u0644\u0645\u064a \u0645\u0639\u0647\u0627 \u0645\u0628\u0627\u0634\u0631\u0629 \u0628\u0636\u0645\u064a\u0631 \u0627\u0644\u0645\u062e\u0627\u0637\u0628\u0629:
1. \u0627\u0644\u062a\u0639\u0627\u0637\u0641: \u0627\u0639\u062a\u0631\u0641\u064a \u0628\u0645\u0627 \u062a\u062d\u0633 \u0628\u0647 \u0648\u0633\u0645\u0651\u064a\u0647 \u0628\u0645\u0635\u0637\u0644\u062d\u0647 \u0627\u0644\u0646\u0641\u0633\u064a \u0627\u0644\u0635\u062d\u064a\u062d
2. \u0627\u0644\u062a\u0641\u0633\u064a\u0631 \u0627\u0644\u0646\u0641\u0633\u064a: \u0627\u0634\u0631\u062d\u064a\u0644\u0647\u0627 \u0645\u0627 \u062a\u0645\u0631 \u0628\u0647 (\u0627\u0644\u0644\u064a \u062a\u062d\u0633\u064a \u0628\u064a\u0647 \u064a\u0633\u0645\u0649...)
3. \u0627\u0644\u0646\u0635\u0627\u0626\u062d \u0627\u0644\u0639\u0645\u0644\u064a\u0629: \u062e\u0637\u0648\u0627\u062a \u0645\u0644\u0645\u0648\u0633\u0629 \u0648\u0642\u0627\u0628\u0644\u0629 \u0644\u0644\u062a\u0637\u0628\u064a\u0642 \u0641\u0648\u0631\u0627\u064b
4. \u0627\u0644\u0645\u0648\u0627\u0631\u062f: \u0631\u0642\u0645 1899 \u0625\u0630\u0627 \u0643\u0627\u0646\u062a \u0641\u064a \u062e\u0637\u0631
5. \u0631\u0633\u0627\u0644\u0629 \u0623\u0645\u0644 \u0648\u062a\u0634\u062c\u064a\u0639 \u0634\u062e\u0635\u064a\u0629"""

SYSTEM_PROMPT_FR = """Tu es une psychologue clinicienne professionnelle specialisee dans l'accompagnement
des femmes victimes de violence conjugale, psychologique, sexuelle et economique.
Tu parles en francais clair, chaleureux, non-jugeant et professionnel.

Regles absolues :
- Zero terme informatique (RAG, LLM, AI, algorithme...)
- Francais uniquement - jamais anglais
- Ne blame JAMAIS la victime, sous aucune forme
- Utilise des termes psychologiques precis et expliques :
  traumatisme, SSPT, anxiete, depression, violence psychologique,
  emprise emotionnelle, syndrome de Stockholm, gaslighting,
  dissociation, hypervigilance, cycle de la violence, co-dependance

Structure de chaque reponse (tu t'adresses directement a elle) :
1. Empathie & validation : nomme ce qu'elle ressent avec le terme psychologique exact
2. Analyse psychologique : explique ce qu'elle vit
   (Ce que vous ressentez s'appelle [terme]. C'est une reaction normale car...)
3. Conseils pratiques concrets : etapes immediates et applicables
4. Ressources : 3919 France ou 1899 Tunisie si danger
5. Message d'espoir personnalise et encourageant"""

print('\u2705 Prompts psychologiques professionnels charges.')


✅ Prompts psychologiques professionnels charges.


## ⚙️ Cellule 9F – Génération de réponse (RAG + LLM psychologique)

In [13]:
def generate_response_multilang(question: str, lang: str, history: list) -> str:
    """
    Pipeline complet :
    1. Retrieval RAG (contexte psy uniquement)
    2. Historique de conversation (continuite)
    3. Prompt strict psychologique
    4. Generation LLM avec parametres anti-hallucination
    5. Fallback si reponse vide ou trop courte
    """
    # Etape 1 : Retrieval RAG
    contexts = retrieve(question, top_k=3)
    context_text = '\n\n'.join([
        f"[Reference {i+1}] {c['passage']}" for i, c in enumerate(contexts)
    ]) if contexts else ''

    # Etape 2 : Historique de conversation
    history_text = ''
    if history and len(history) >= 2:
        for msg in history[-6:]:
            role    = 'La femme' if msg['role'] == 'user' else 'Psychologue'
            content = msg['content'].replace('\U0001f3a4 ', '').replace('\u2328\ufe0f ', '').replace('\U0001f338 ', '')
            history_text += f"{role}: {content}\n"

    # Etape 3 : Prompt
    system = SYSTEM_PROMPT_AR if lang == 'ar' else SYSTEM_PROMPT_FR

    if lang == 'ar':
        ctx_block  = f"\u0645\u0639\u0644\u0648\u0645\u0627\u062a \u0646\u0641\u0633\u064a\u0629:\n{context_text}\n" if context_text else ''
        hist_block = f"\u0633\u064a\u0627\u0642 \u0627\u0644\u0645\u062d\u0627\u062f\u062b\u0629:\n{history_text}\n" if history_text else ''
        prompt = (
            f"<s>[INST] {system}\n\n"
            f"\u062a\u0639\u0644\u064a\u0645\u0627\u062a: \u0623\u062c\u064a\u0628\u064a \u0628\u0627\u0644\u062f\u0627\u0631\u062c\u0629 \u0627\u0644\u062a\u0648\u0646\u0633\u064a\u0629 \u0641\u0642\u0637. \u0644\u0627 \u0625\u0646\u062c\u0644\u064a\u0632\u064a\u0629. \u0644\u0627 \u0645\u0635\u0637\u0644\u062d\u0627\u062a \u062a\u0642\u0646\u064a\u0629.\n\n"
            f"{ctx_block}{hist_block}"
            f"\u0645\u0627 \u0642\u0627\u0644\u062a\u0647 \u0627\u0644\u0645\u0631\u0623\u0629: {question}\n\n"
            f"\u0623\u062c\u064a\u0628\u064a \u0627\u0644\u0622\u0646 \u0645\u0628\u0627\u0634\u0631\u0629 \u0645\u0639\u0647\u0627:\n[/INST]"
        )
    else:
        ctx_block  = f"References psychologiques:\n{context_text}\n" if context_text else ''
        hist_block = f"Contexte de la conversation:\n{history_text}\n" if history_text else ''
        prompt = (
            f"<s>[INST] {system}\n\n"
            f"Instructions : francais uniquement. Zero terme informatique.\n\n"
            f"{ctx_block}{hist_block}"
            f"Ce que la femme a dit : {question}\n\n"
            f"Reponds maintenant directement a elle :\n[/INST]"
        )

    # Etape 4 : Generation LLM
    output    = llm_pipe(prompt, max_new_tokens=600, temperature=0.65,
                         repetition_penalty=1.3, do_sample=True, top_p=0.9)
    generated = output[0]['generated_text']
    response  = generated.split('[/INST]')[-1].strip() \
                if '[/INST]' in generated else generated.strip()

    # Etape 5 : Nettoyage + fallback
    response = re.sub(r'\n{3,}', '\n\n', response)
    if len(response.strip()) < 30:
        if lang == 'ar':
            response = '\u0623\u0646\u0627 \u0647\u0646\u0627 \u0645\u0639\u0627\u0643\u0650 \u0648\u0623\u0633\u0645\u0639\u0643. \u062d\u0643\u064a\u0644\u064a \u0623\u0643\u062b\u0631 \u0639\u0644\u0649 \u0627\u0644\u0644\u064a \u062a\u062d\u0633\u064a \u0628\u064a\u0647 \u062d\u062a\u0649 \u0646\u0642\u062f\u0631 \u0646\u0633\u0627\u0639\u062f\u0643 \u0623\u0643\u062b\u0631.'
        else:
            response = "Je suis la pour vous et je vous ecoute. Pouvez-vous me dire davantage ce que vous ressentez ?"

    return response


print('\u2705 Generation psychologique professionnelle prete.')


✅ Generation psychologique professionnelle prete.


## 🔄 Cellule 9G – Pipeline principal (process_input)

In [14]:
def process_input(audio_input, text_input, response_mode, history):
    history = history or []

    if audio_input is not None:
        try:
            user_text, lang = transcribe_audio(audio_input)
            if not user_text:
                return history, None, '', '\u26a0\ufe0f Audio vide. Reessayez.'
            user_label        = f'\U0001f3a4 {user_text}'
            transcription_msg = f'\U0001f4dd Whisper : {user_text} ({"\U0001f1f9\U0001f1f3 Darija" if lang == "ar" else "\U0001f1eb\U0001f1f7 Francais"})'
        except Exception as e:
            return history, None, '', f'\u274c Erreur transcription : {str(e)}'
    elif text_input and text_input.strip():
        user_text         = text_input.strip()
        lang              = detect_language(user_text)
        user_label        = f'\u2328\ufe0f {user_text}'
        transcription_msg = ''
    else:
        return history, None, '', '\u26a0\ufe0f Veuillez saisir un message ou enregistrer un audio.'

    try:
        response_text = generate_response_multilang(user_text, lang, history)
    except Exception as e:
        response_text = f'\u274c Erreur generation : {str(e)}'

    audio_out = None
    if response_mode == '\U0001f50a Vocale':
        try:
            audio_out = text_to_speech(response_text, lang)
        except Exception as e:
            print(f'\u26a0\ufe0f TTS error: {e}')

    history.append({'role': 'user',      'content': user_label})
    history.append({'role': 'assistant', 'content': f'\U0001f338 {response_text}'})

    return history, audio_out, '', transcription_msg


print('\u2705 Pipeline process_input() pret.')


✅ Pipeline process_input() pret.


## 🌸 Cellule 9H – Interface Gradio & Lancement

In [15]:
with gr.Blocks(title='Assistant Psychologique', css=CSS) as demo:

    gr.HTML("<h1 id='title'>\U0001f338 Espace Ecoute & Soutien / \u0641\u0636\u0627\u0621 \u0627\u0644\u0627\u0633\u062a\u0645\u0627\u0639 \u0648\u0627\u0644\u062f\u0639\u0645</h1>")
    gr.HTML("<p id='subtitle'>Parlez en francais ou en tunisien - \u062a\u0642\u062f\u0631\u064a \u062a\u062d\u0643\u064a \u0628\u0627\u0644\u0641\u0631\u0646\u0633\u064a\u0629 \u0623\u0648 \u0628\u0627\u0644\u062f\u0627\u0631\u062c\u0629</p>")
    gr.Markdown(DISCLAIMER, elem_classes=['disclaimer'])

    with gr.Row():
        with gr.Column(scale=3):

            chatbot = gr.Chatbot(
                height=450, type='messages',
                avatar_images=(None, '\U0001f338'),
                label='Conversation',
                rtl=True, show_copy_button=True,
            )

            gr.HTML("<div class='response-mode-box'>\U0001f3db\ufe0f <b>Mode de reponse / \u0637\u0631\u064a\u0642\u0629 \u0627\u0644\u062c\u0648\u0627\u0628</b></div>")
            response_mode = gr.Radio(
                choices=['\U0001f4dd Texte', '\U0001f50a Vocale'],
                value='\U0001f4dd Texte',
                label="L'IA repond par",
                info='Lire ou ecouter la reponse'
            )

            audio_response = gr.Audio(
                label='\U0001f50a Reponse vocale',
                autoplay=True, type='filepath', visible=False
            )
            response_mode.change(
                fn=lambda m: gr.update(visible=(m == '\U0001f50a Vocale')),
                inputs=[response_mode], outputs=[audio_response]
            )

            with gr.Tabs():

                with gr.Tab('\U0001f3a4 Parler / \u062d\u0643\u064a'):
                    gr.HTML(MIC_WARNING)
                    audio_input = gr.Audio(
                        sources=['microphone', 'upload'],
                        type='filepath',
                        label='\U0001f399\ufe0f Message vocal',
                        format='mp3',
                    )
                    voice_status = gr.Textbox(
                        label='\U0001f4dd Transcription Whisper',
                        interactive=False,
                        placeholder='Ce que Whisper a compris...'
                    )
                    voice_btn = gr.Button('\U0001f3a4 Envoyer', variant='primary', size='lg')
                    voice_btn.click(
                        fn=process_input,
                        inputs=[audio_input, gr.Textbox(visible=False, value=''), response_mode, chatbot],
                        outputs=[chatbot, audio_response, gr.Textbox(visible=False), voice_status]
                    )

                with gr.Tab('\u2328\ufe0f Ecrire / \u0643\u062a\u0627\u0628\u0629'):
                    text_input = gr.Textbox(
                        placeholder='Ecrivez en francais ou \u0628\u0627\u0644\u062f\u0627\u0631\u062c\u0629 \u0627\u0644\u062a\u0648\u0646\u0633\u064a\u0629...',
                        lines=3, label='Votre message'
                    )
                    text_btn    = gr.Button('\U0001f4ac Envoyer', variant='primary', size='lg')
                    text_status = gr.Textbox(visible=False)
                    text_input.submit(
                        fn=process_input,
                        inputs=[gr.Audio(visible=False), text_input, response_mode, chatbot],
                        outputs=[chatbot, audio_response, text_input, text_status]
                    )
                    text_btn.click(
                        fn=process_input,
                        inputs=[gr.Audio(visible=False), text_input, response_mode, chatbot],
                        outputs=[chatbot, audio_response, text_input, text_status]
                    )
                    gr.Examples(
                        examples=EXAMPLES_FR + EXAMPLES_TN,
                        inputs=text_input,
                        label='\U0001f4a1 Exemples de situations'
                    )

                with gr.Tab('\U0001f5d1\ufe0f Nouvelle conversation'):
                    clear_btn = gr.Button('\U0001f5d1\ufe0f Effacer', variant='stop')
                    clear_btn.click(lambda: [], outputs=[chatbot])

        with gr.Column(scale=1):
            gr.Markdown('### \U0001f4de Urgences')
            gr.Markdown("""
| \U0001f30d | Numero |
|----|--------|
| \U0001f1f9\U0001f1f3 Tunisie | **1899** |
| \U0001f1eb\U0001f1f7 France | **3919** |
| \U0001f1f2\U0001f1e6 Maroc | **8007** |
| \U0001f1e7\U0001f1ea Belgique | **0800 30 030** |
            """)
            gr.Markdown('### \U0001f49c Themes abordes')
            gr.Markdown("""
- Violence physique
- Violence psychologique
- Gaslighting & emprise
- Violence sexuelle
- Violence economique
- Strategies de coping
- Plan de securite
- Resilience & guerison
            """)
            gr.Markdown('### \U0001f527 Fix micro')
            gr.Markdown("""
1. \U0001f512 Cadenas dans l'URL
2. Microphone → Autoriser
3. \U0001f504 Rechargez la page
4. Ou uploadez un fichier audio
            """)

demo.queue(max_size=10)
demo.launch(share=True, debug=True, show_error=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8c1a0e3fefa2af9ce8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'top_p', 'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8c1a0e3fefa2af9ce8.gradio.live


## 📊 Cellule 10 – Évaluation du Retrieval RAG

In [16]:
eval_queries = [
    ('culpabilite apres agression',       'violence sexuelle / coping'),
    ('mon mari controle mon argent',       'violence economique'),
    ('cauchemars flashbacks trauma',       'SSPT / therapie'),
    ('comment quitter en securite',        'plan de securite'),
    ('signes d une relation toxique',      'signaux d alerte'),
    ('enfants temoins de disputes',        'enfants / violence'),
    ('je doute de moi tout le temps',      'gaslighting'),
    ('exercices pour calmer l angoisse',   'grounding / coping'),
]

print('\U0001f4ca EVALUATION DU RETRIEVAL')
print('='*70)
scores_list = []
for query, expected in eval_queries:
    results   = retrieve(query, top_k=1)
    top_score = results[0]['score']
    snippet   = results[0]['passage'][:80] + '...'
    scores_list.append(top_score)
    icon = '\u2705' if top_score > 0.5 else ('\u26a0\ufe0f' if top_score > 0.35 else '\u274c')
    print(f"\n{icon} [{top_score:.3f}] Requete: '{query}'")
    print(f'   Theme attendu  : {expected}')
    print(f'   Passage trouve : {snippet}')

print('\n' + '='*70)
print(f'Score moyen : {sum(scores_list)/len(scores_list):.3f}')
print('Legende : \u2705 > 0.50 | \u26a0\ufe0f > 0.35 | \u274c <= 0.35')


📊 EVALUATION DU RETRIEVAL

✅ [0.681] Requete: 'culpabilite apres agression'
   Theme attendu  : violence sexuelle / coping
   Passage trouve : La violence psychologique inclut l'humiliation, le dénigrement, l'isolement et l...

❌ [0.326] Requete: 'mon mari controle mon argent'
   Theme attendu  : violence economique
   Passage trouve : إذا كان زوجك يتحكم في فلوسك ويمنعك من العمل فهذا يسمى العنف الاقتصادي.
    حقك ف...

✅ [0.629] Requete: 'cauchemars flashbacks trauma'
   Theme attendu  : SSPT / therapie
   Passage trouve : Le gaslighting amène la victime à douter de sa propre mémoire et de ses percepti...

⚠️ [0.427] Requete: 'comment quitter en securite'
   Theme attendu  : plan de securite
   Passage trouve : Quitter une relation violente est l'un des moments les plus dangereux.
    Prépa...

✅ [0.531] Requete: 'signes d une relation toxique'
   Theme attendu  : signaux d alerte
   Passage trouve : Signes d'alerte : jalousie excessive présentée comme amour, isolement progressif...

✅

## ➕ Cellule 11 – Enrichissement dynamique de la base RAG

In [17]:
def add_documents(new_docs: list):
    """Ajoute de nouveaux passages a l'index FAISS dynamiquement."""
    global KNOWLEDGE_BASE, corpus_embeddings
    new_emb = embed_model.encode(
        new_docs, convert_to_numpy=True, normalize_embeddings=True
    ).astype(np.float32)
    index.add(new_emb)
    KNOWLEDGE_BASE.extend(new_docs)
    corpus_embeddings = np.vstack([corpus_embeddings, new_emb])
    print(f'\u2705 {len(new_docs)} doc(s) ajoute(s). Total : {index.ntotal} passages.')


additional_docs = [
    """La therapie par l'art permet aux survivantes d'exprimer des emotions difficiles.
    Peinture, sculpture, musique ou ecriture creative offrent des voies alternatives
    pour traiter le trauma et reconstruire une identite positive.""",

    """Le droit a l'ordonnance de protection permet d'obtenir l'eloignement de l'agresseur
    par decision judiciaire, meme sans depot de plainte prealable.
    Contactez un avocat ou une association pour vous accompagner.""",
]

add_documents(additional_docs)


✅ 2 doc(s) ajoute(s). Total : 24 passages.


---
## 🔧 Améliorations possibles

| Axe | Amélioration |
|-----|--------------|
| **Données** | Intégrer des PDF de guides cliniques (OMS, UNHCR) via PyPDF2 |
| **Retrieval** | Utiliser un reranker cross-encoder pour affiner FAISS |
| **LLM** | Fine-tuner sur des dialogues de soutien psychologique réels |
| **Sécurité** | Détection automatique des situations de danger immédiat |
| **Évaluation** | Métriques RAGAS (faithfulness, answer_relevancy) |

---
🌸 **Chaque femme mérite d'être entendue, crue et soutenue.**
